<a href="https://colab.research.google.com/github/yashas-anil/election-flippers/blob/main/2024_election_flippers.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# You Can Flip Any Election by Moving People Around: 2024

The 2024 election between Republican Donald Trump and Democrat Kamala Harris had an EV margin of 86 (312-226) and a popular vote margin of 2.3 million, or 1.5% (49.8-48.3). Not a landslide but not especially close; Trump carried every state considered a tossup.

# Data Source

In [2]:
import pandas as pd

In [3]:
# data source: 2024 election results, wikipedia
url = "https://en.wikipedia.org/wiki/2024_United_States_presidential_election"

In [4]:
# for some reason there was an http error so I have to pretend to be a browser
import requests

header = {
  "User-Agent": "Mozilla/5.0 (X11; Linux x86_64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/50.0.2661.75 Safari/537.36",
  "X-Requested-With": "XMLHttpRequest"
}

r = requests.get(url, headers=header)

tables = pd.read_html(r.text) # array of all dataframes

C:\Users\yasha\AppData\Local\Temp\ipykernel_2064\3480215772.py:11: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  tables = pd.read_html(r.text) # array of all dataframes


In [5]:
raw_df = tables[24]

# Data Transformation
1. Cleaning the States column, dropping unneeded columns/rows, renaming columns for simplicity
2. Convert all numeric columns from string to int
3. Extract and drop national results
4. Add VPP column

In [16]:
raw_df.columns[4][0].split()[0]

'Harris/Walz'

In [ ]:
df = raw_df.copy(deep=True)

# remove the [] artifacts
import re

# Example column name
col = "State or district"
states = df[col][col]


# Remove square brackets and anything inside them
states = (
    states
    .str.replace(r"\[.*?\]", "", regex=True)          # remove [..]
    .str.replace(r"Tooltip.*", "", regex=True)        # remove Tooltip + following text
    .str.replace("†", "", regex=False)                # remove dagger symbol
    .str.strip()                                      # remove leading/trailing spaces
)


df.insert(1, "States", states)
df=df.drop("State or district", axis=1)

# remove unneeded columns, row, renamed for ease
df=df.drop(["Stein/Ware Green","Oliver/Maat Libertarian","Kennedy/Shanahan Independent","Others","Margin swing[b]"],axis=1)
df=df.drop(57)


ogwin = df.columns[1][0]
oglos = df.columns[4][0]
# df = df.rename(columns={"Trump/Vance Republican": ogwin, 'Harris/Walz Democratic': oglos})


C:\Users\yasha\AppData\Local\Temp\ipykernel_2064\855495978.py:22: PerformanceWarning: dropping on a non-lexsorted multi-index without a level parameter may impact performance.
  df=df.drop("State or district", axis=1)
C:\Users\yasha\AppData\Local\Temp\ipykernel_2064\855495978.py:25: PerformanceWarning: dropping on a non-lexsorted multi-index without a level parameter may impact performance.
  df=df.drop(["Stein/Ware Green","Oliver/Maat Libertarian","Kennedy/Shanahan Independent","Others","Margin swing[b]"],axis=1)


States Trump/Vance Republican             Harris/Walz Democratic  \
                               Votes       %  EV                  Votes   
0     Alabama                1462616  64.57%   9                 772412   
1      Alaska                 184458  54.54%   3                 140026   
2     Arizona                1770242  52.22%  11                1582860   
3    Arkansas                 759241  64.20%   6                 396905   
4  California                6081697  38.33%   –                9276179   

                   Margin          Total votes  
        %  EV       Votes        % Total votes  
0  34.10%   –      690204   30.47%     2265090  
1  41.41%   –       44432   13.13%      338177  
2  46.69%   –      187382    5.53%     3390161  
3  33.56%   –      362336   30.64%     1182676  
4  58.47%  54  −3,194,482  −20.14%    15865475

In [19]:
# convert votes, %, EV from str to int
# apparently the minus signs are not the correct symbol, which is tripping up the str to int coversion, so we have to fix those dashes

num_df=df.copy(deep=True) # create new dataframe for simplicity

DASHES = "\u2212\u2012\u2013\u2014\u2015"  # minus, figure dash, en dash, em dash, horiz bar
num_cols=[ogwin,oglos,"Margin"]
for col in num_cols:
  s=df[col]['Votes']
  s = (
    s.str.strip()
    # Normalize any Unicode dash to ASCII hyphen-minus
    .str.replace(f"[{DASHES}]", "-", regex=True)
    # Remove commas used as thousands separators
    .str.replace(",", "", regex=False)
  )
  nums = pd.to_numeric(s, errors="coerce")

  # Convert to nullable integer type
  num_df.loc[:,(col,'Votes')] = nums.astype("Int64") # truncates toward 0 (e.g., -1.9 -> -1)

  # %
  p=df[col]['%']
  p = (
    p.str.strip()
    # Normalize any Unicode dash to ASCII hyphen-minus
    .str.replace(f"[{DASHES}]", "-", regex=True)
    # Remove %
    .str.replace("%", "", regex=False)
  )
  nums = pd.to_numeric(p, errors="coerce")

  # Convert to nullable integer type
  num_df.loc[:,(col,'%')] = nums.astype("Float64") # truncates toward 0 (e.g., -1.9 -> -1)


  #  EV
  if col!='Margin': # margin doesnt have an EV col
    e=df[col]['EV']
    e=e.str.replace(f"[{DASHES}]", "0", regex=True) # the "nulls" are just dashes so we can replace them with 0s
    nums = pd.to_numeric(e, errors="coerce")
    num_df.loc[:,(col,'EV')] = nums.astype("Int64")

#convert Total
num_df.rename(columns={"Total votes":"Total"},inplace=True)
col="Total votes"
s=df[col][col]
nums = pd.to_numeric(s, errors="coerce")
num_df['Total Votes'] = nums.astype("Int64")
num_df=num_df.drop("Total", axis=1) #drop the old total

num_df

C:\Users\yasha\AppData\Local\Temp\ipykernel_2064\2632305870.py:50: PerformanceWarning: dropping on a non-lexsorted multi-index without a level parameter may impact performance.
  num_df=num_df.drop("Total", axis=1) #drop the old total


States Trump/Vance Republican             Harris/Walz Democratic  \
                                    Votes      %   EV                  Votes   
0          Alabama                1462616  64.57    9                 772412   
1           Alaska                 184458  54.54    3                 140026   
2          Arizona                1770242  52.22   11                1582860   
3         Arkansas                 759241   64.2    6                 396905   
4       California                6081697  38.33    0                9276179   
5         Colorado                1377441  43.14    0                1728159   
6      Connecticut                 736918  41.89    0                 992053   
7         Delaware                 214351  41.79    0                 289758   
8             D.C.                  21076   6.47    0                 294185   
9          Florida                6110125  56.09   30                4683038   
10         Georgia                2663117  50.72   16                2548017   
11          Hawaii                 193661  37.48    0                 313044   
12           Idaho                 605246  66.87    4                 274972   
13        Illinois                2449079  43.47    0                3062863   
14         Indiana                1720347  58.58   11                1163603   
15            Iowa                 927019  55.73    6                 707278   
16          Kansas                 758802  57.16    6                 544853   
17        Kentucky                1337494  64.47    8                 704043   
18       Louisiana                1208505  60.22    8                 766870   
19           Maine                 377977  45.46    0                 435652   
20            ME-1                 165214  38.09    0                 258863   
21            ME-2                 212763   53.5    1                 176789   
22        Maryland                1035550  34.08    0                1902577   
23   Massachusetts                1251303  36.02    0                2126518   
24        Michigan                2816636  49.73   15                2736533   
25       Minnesota                1519032  46.68    0                1656979   
26     Mississippi                 747744  60.89    6                 466668   
27        Missouri                1751986  58.49   10                1200599   
28         Montana                 352079  58.39    4                 231906   
29        Nebraska                 564816  59.32    2                 369995   
30            NE-1                 177666  55.49    1                 136153   
31            NE-2                 148905  46.73    0                 163541   
32            NE-3                 238245  76.03    1                  70301   
33          Nevada                 751205  50.59    6                 705197   
34   New Hampshire                 395523  47.87    0                 418488   
35      New Jersey                1968215  46.06    0                2220713   
36      New Mexico                 423391  45.85    0                 478802   
37        New York                3578899  43.31    0                4619195   
38  North Carolina                2898423  50.86   16                2715375   
39    North Dakota                 246505  66.96    3                 112327   
40            Ohio                3180116  55.14   17                2533699   
41        Oklahoma                1036213  66.16    7                 499599   
42          Oregon                 919480  40.97    0                1240600   
43    Pennsylvania                3543308  50.37   19                3423042   
44    Rhode Island                 214406  41.76    0                 285156   
45  South Carolina                1483747  58.23    9                1028452   
46    South Dakota                 272081  63.43    3                 146859   
47       Tennessee                1966865  64.19   11                1056265   
48          

In [22]:
# the total row is the popular vote, so we'll extract that and keep it separate
national=num_df.iloc[56]
num_df=num_df.drop(56)
national

States                               Total
Trump/Vance Republican  Votes     77302580
                        %             49.8
                        EV             312
Harris/Walz Democratic  Votes     75017613
                        %            48.32
                        EV             226
Margin                  Votes      2284967
                        %             1.48
Total Votes                      155238302
VPP                                4247.15
Name: 56, dtype: object

Voters per Point (VPP) is a metric to determine whether a state should be flipped. It is defined as Margin(voters)/StateEV, and tells us how many voters each flipped EV point "costs". Therefore we want to start our flipping algorithm on those with the lowest VPP, which have the most value for their cost. Additionally, when using an underflow algorithm, we want to sacrifice our high VPP states, because they are the least worth flipping.

In [20]:
num_df['VPP']=num_df["Margin"]['Votes']/(num_df[oglos]['EV']+num_df[ogwin]['EV'])
num_df['VPP']=num_df['VPP'].astype(float).round(2)
# sort_df=num_df.sort_values(by='VPP')

# Flipping Algorithm
1.   Calculate Minimum EV needed to flip
2.   Split into winner and loser dfs

In [23]:
# Minimum EV
import math
nat_total_EV=national[(ogwin,"EV")]+national[(oglos,"EV")]
if nat_total_EV%2==0:
  threshold_EV=int(nat_total_EV/2)+1
else:
  threshold_EV=math.ceil(nat_total_EV/2)
minimum_EV=threshold_EV-national[(oglos,"EV")] #the minimum EV to flip to flip the entire election
minimum_EV

44

In [33]:
losers_df=num_df[num_df['Margin']['Votes']<0]
losers_df=losers_df.sort_values(by=[('Margin',"Votes")])
losers_df.reset_index(inplace=True) #reset index to put low VPP at top
losers_df.drop("index", axis=1, inplace=True) #drop original mixed up index
losers_df.head()

C:\Users\yasha\AppData\Local\Temp\ipykernel_2064\1885206875.py:4: PerformanceWarning: dropping on a non-lexsorted multi-index without a level parameter may impact performance.
  losers_df.drop("index", axis=1, inplace=True) #drop original mixed up index


States Trump/Vance Republican           Harris/Walz Democratic  \
                                  Votes      % EV                  Votes   
0     California                6081697  38.33  0                9276179   
1       New York                3578899  43.31  0                4619195   
2  Massachusetts                1251303  36.02  0                2126518   
3       Maryland                1035550  34.08  0                1902577   
4     Washington                1530923  39.01  0                2245849   

                Margin        Total Votes       VPP  
       %  EV     Votes      %                        
0  58.47  54  -3194482 -20.14    15865475 -59157.07  
1  55.91  28  -1040296  -12.6     8262495 -37153.43  
2  61.22  11   -875215  -25.2     3473668 -79565.00  
3  62.62  10   -867027 -28.54     3038334 -86702.70  
4  57.23  12   -714926 -18.22     3924243 -59577.17

In [34]:
winners_df=num_df[num_df['Margin']['Votes']>0]
winners_df=winners_df.sort_values(by='VPP')
winners_df.reset_index(inplace=True) #reset index to put low VPP at top
winners_df.drop("index", axis=1, inplace=True) #drop original mixed up index
winners_df.head()

C:\Users\yasha\AppData\Local\Temp\ipykernel_2064\2797676811.py:4: PerformanceWarning: dropping on a non-lexsorted multi-index without a level parameter may impact performance.
  winners_df.drop("index", axis=1, inplace=True) #drop original mixed up index


States Trump/Vance Republican            Harris/Walz Democratic  \
                                 Votes      %  EV                  Votes   
0     Wisconsin                1697626   49.6  10                1668229   
1      Michigan                2816636  49.73  15                2736533   
2  Pennsylvania                3543308  50.37  19                3423042   
3       Georgia                2663117  50.72  16                2548017   
4        Nevada                 751205  50.59   6                 705197   

             Margin       Total Votes      VPP  
       % EV   Votes     %                       
0  48.74  0   29397  0.86     3422918  2939.70  
1  48.31  0   80103  1.42     5664186  5340.20  
2  48.66  0  120266  1.71     7058732  6329.79  
3  48.53  0  115100  2.19     5250905  7193.75  
4  47.49  0   46008   3.1     1484840  7668.00

# Overflow Algorithm: Minimum EV
The primary source state is the state from the og loser with the most votes to spare (overflow). In 2024 it was California.

Because the outcome is flipped, the copy of losers will be called "flip winners" and the copy of winners will be "flip losers"

Specifically, as this first algorithm will just flip the minimum EV and will use overflow method, the copies will be called mow_df (min_over_winners) and mol_df (min_over_losers)

In [27]:
mow_df=losers_df.copy(deep=True)

mol_df=winners_df.copy(deep=True)

sources=1 #number of source states
flipped_EV=0
total_voters_moved=0 #the number of voters moved overall
state_voters_moved=0 #the number of voters moved into each state to flip it
mini_margin=1000  #our arbitrary minimum acceptable margin of statewide victory

i=0 # source state index
j=0 # target state index
pover=0 # if a state's overflow is less than the margin, carry over to the next source state's overflow

source_state=mow_df.iloc[i] #the state we are moving voters from
target_state=mol_df.iloc[j] #the state we are moving voters to
overflow=source_state.loc[("Margin","Votes")]*(-1)-mini_margin+pover # the number of voters we have to move from the source state, plus any leftover voters we had to carry over from the previous source state


while i<sources and j<len(mol_df): # while we still have source states and target states to move voters from/to

  if overflow>(target_state.loc[("Margin","Votes")]+mini_margin): #if we have enough voters in the source state to flip the target state with a mini_margin, move the number of voters needed to flip the target state with a mini_margin
    pover=0 # reset pover since we have enough voters to flip the target state
    
    state_voters_moved=target_state.loc[("Margin","Votes")]+mini_margin
    # change votes
    mol_df.loc[j,(oglos,"Votes")]=int(mol_df.loc[j,(oglos,"Votes")])+state_voters_moved
    mol_df.loc[j,"Total Votes"]=int(mol_df["Total Votes"].iloc[j])+state_voters_moved
    mol_df.loc[j,("Margin","Votes")]=mini_margin*(-1)

    # change EV
    flipped_EV=flipped_EV+mol_df.loc[j,(ogwin,"EV")]
    mol_df.loc[j,(oglos,"EV")]=mol_df.loc[j,(ogwin,"EV")]
    mol_df.loc[j,(ogwin,"EV")]=0
    
    # change overflow
    overflow=overflow-state_voters_moved
    total_voters_moved=total_voters_moved+state_voters_moved # update total voters moved
    
    print(f"Flipped {target_state.loc['States'].to_string()[4:]} by moving {state_voters_moved} voters from {source_state.loc['States'].to_string()[4:]}. Total voters moved: {total_voters_moved}. Flipped EV: {flipped_EV}. Overflow: {overflow}.")
    if flipped_EV>=minimum_EV:
      print(f"Flipped election! Flipped EV: {flipped_EV}. Minimum EV needed: {minimum_EV}. Total voters moved: {total_voters_moved}.")
      break
    
    # move on to the next target state
    j=j+1
    target_state=mol_df.iloc[j] 

  else: # store overflow in pover and move on to the next source state
    print(f"Not enough voters in {source_state.loc['States'].to_string()[4:]} to flip {target_state.loc['States'].to_string()[4:]}. Overflow: {overflow}. Moving on to the next source state.")
    pover=overflow
    i+=1 
    source_state=mow_df.iloc[i] if i<sources else None
    overflow=source_state.loc[("Margin","Votes")]*(-1)-mini_margin+pover if i<sources else None # update overflow for the new source state




# mol_df

Flipped Wisconsin by moving 30397 voters from California. Total voters moved: 30397. Flipped EV: 10. Overflow: 3163085.
Flipped Michigan by moving 81103 voters from California. Total voters moved: 111500. Flipped EV: 25. Overflow: 3081982.
Flipped Pennsylvania by moving 121266 voters from California. Total voters moved: 232766. Flipped EV: 44. Overflow: 2960716.
Flipped election! Flipped EV: 44. Minimum EV needed: 44. Total voters moved: 232766.


In [29]:
# update the source states
mow_df=losers_df.copy(deep=True)

i=0
reimburse = total_voters_moved
while i<sources:
    if reimburse>mow_df.loc[i,("Margin","Votes")]*(-1)-mini_margin:
        state_voters_removed=mow_df.loc[i,("Margin","Votes")]*(-1)-mini_margin

        # change votes
        mow_df.loc[i,(oglos,"Votes")]=int(mow_df.loc[i,(oglos,"Votes")])-state_voters_removed
        mow_df.loc[i,"Total Votes"]=int(mow_df["Total Votes"].iloc[i])-state_voters_removed
        mow_df.loc[i,("Margin","Votes")]=mini_margin*(-1)

        reimburse-=state_voters_removed
        print(f"Removed {state_voters_removed} voters from {mow_df.loc[i,'States'].to_string()[4:]}. Remaining voters to reimburse: {reimburse}.")
    else:
        state_voters_removed=reimburse

        # change votes
        mow_df.loc[i,(oglos,"Votes")]=int(mow_df.loc[i,(oglos,"Votes")])-state_voters_removed
        mow_df.loc[i,"Total Votes"]=int(mow_df["Total Votes"].iloc[i])-state_voters_removed
        mow_df.loc[i,("Margin","Votes")]=mow_df.loc[i,("Margin","Votes")]+state_voters_removed

        reimburse-=state_voters_removed
        print(f"Removed {state_voters_removed} voters from {mow_df.loc[i,'States'].to_string()[4:]}. All voters reimbursed.")
    i+=1
        
# mow_df

Removed 232766 voters from California. All voters reimbursed.


In [30]:
# concatenate the winner and loser
mfinal_df=pd.concat([mol_df,mow_df]).sort_values(by=[("Margin","Votes")])

#update percentages
mfinal_df.loc[:,(oglos,"%")]=(mfinal_df.loc[:,(oglos,"Votes")]/mfinal_df.loc[:,"Total Votes"]).astype(float).round(4)*100
mfinal_df.loc[:,(ogwin,"%")]=(mfinal_df.loc[:,(ogwin,"Votes")]/mfinal_df.loc[:,"Total Votes"]).astype(float).round(4)*100
mfinal_df.loc[:,("Margin","%")]=(mfinal_df.loc[:,("Margin","Votes")]/mfinal_df.loc[:,"Total Votes"]).astype(float).round(4)*100

mfinal_df

States Trump/Vance Republican            Harris/Walz Democratic  \
                                    Votes      %  EV                  Votes   
0       California                6081697   38.9   0                9043413   
1         New York                3578899  43.31   0                4619195   
2    Massachusetts                1251303  36.02   0                2126518   
3         Maryland                1035550  34.08   0                1902577   
4       Washington                1530923  39.01   0                2245849   
5         Illinois                2449079  43.47   0                3062863   
6         Colorado                1377441  43.14   0                1728159   
7           Oregon                 919480  40.97   0                1240600   
8             D.C.                  21076   6.47   0                 294185   
9         Virginia                2075085  46.05   0                2335395   
10     Connecticut                 736918  41.89   0                 992053   
11      New Jersey                1968215  46.06   0                2220713   
12       Minnesota                1519032  46.68   0                1656979   
13          Hawaii                 193661  37.48   0                 313044   
14         Vermont                 119395  32.32   0                 235791   
15            ME-1                 165214  38.09   0                 258863   
16        Delaware                 214351  41.79   0                 289758   
17    Rhode Island                 214406  41.76   0                 285156   
18           Maine                 377977  45.46   0                 435652   
19      New Mexico                 423391  45.85   0                 478802   
20   New Hampshire                 395523  47.87   0                 418488   
21            NE-2                 148905  46.73   0                 163541   
0        Wisconsin                1697626  49.16   0                1698626   
2     Pennsylvania                3543308  49.35   0                3544308   
1         Michigan                2816636  49.03   0                2817636   
10            ME-2                 212763   53.5   1                 176789   
15            NE-1                 177666  55.49   1                 136153   
6           Alaska                 184458  54.54   3                 140026   
4           Nevada                 751205  50.59   6                 705197   
3          Georgia                2663117  50.72  16                2548017   
8          Montana                 352079  58.39   4                 231906   
14         Wyoming                 192633   71.6   3                  69527   
16    South Dakota                 272081  63.43   3                 146859   
17    North Dakota                 246505  66.96   3                 112327   
33            NE-3                 238245  76.03   1                  70301   
5   North Carolina                2898423  50.86  16                2715375   
7          Arizona                1770242  52.22  11                1582860   
32        Nebraska                 564816  59.32   2                 369995   
9           Kansas                 758802  57.16   6                 544853   
11            Iowa                 927019  55.73   6                 707278   
18     Mississippi                 747744  60.89   6                 466668   
29   West Virginia                 533556  69.97   4                 214309   
22            Utah                 883818  59.38   6                 562566   
30           Idaho                 605246  66.87   4                 274972   
25        Arkansas                 759241   64.2   6                 396905   
24       Louisiana                1208505  60.22   8                 766870   
20  South Carolina                1483747  58.23   9                1028452   
26        Oklahoma                1036213  66.16   7                 499599   
23        Missouri                1751986  58.49  10          

# Overflow: Multiple Source States

In [35]:
ow_df=losers_df.copy(deep=True)

ol_df=winners_df.copy(deep=True)

# ow_df #so we can compare it to the outcome of the algo

In [ ]:
ow_df=losers_df.copy(deep=True)

ol_df=winners_df.copy(deep=True)

sources=1 #number of source states: CUSTOMIZABLE

flipped_EV=0
total_voters_moved=0 #the number of voters moved overall
state_voters_moved=0 #the number of voters moved into each state to flip it
mini_margin=1000  #our arbitrary minimum acceptable margin of statewide victory

i=0 # source state index
j=0 # target state index
pover=0 # if a state's overflow is less than the margin, carry over to the next source state's overflow

source_state=ow_df.iloc[i] #the state we are moving voters from
target_state=ol_df.iloc[j] #the state we are moving voters to
overflow=source_state.loc[("Margin","Votes")]*(-1)-mini_margin+pover # the number of voters we have to move from the source state, plus any leftover voters we had to carry over from the previous source state


while i<sources and j<len(ol_df): # while we still have source states and target states to move voters from/to

  if overflow>(target_state.loc[("Margin","Votes")]+mini_margin): #if we have enough voters in the source state to flip the target state with a mini_margin, move the number of voters needed to flip the target state with a mini_margin
    pover=0 # reset pover since we have enough voters to flip the target state
    
    state_voters_moved=target_state.loc[("Margin","Votes")]+mini_margin
    # change votes
    ol_df.loc[j,(oglos,"Votes")]=int(ol_df.loc[j,(oglos,"Votes")])+state_voters_moved
    ol_df.loc[j,"Total Votes"]=int(ol_df["Total Votes"].iloc[j])+state_voters_moved
    ol_df.loc[j,("Margin","Votes")]=mini_margin*(-1)

    # change EV
    flipped_EV=flipped_EV+ol_df.loc[j,(ogwin,"EV")]
    ol_df.loc[j,(oglos,"EV")]=ol_df.loc[j,(ogwin,"EV")]
    ol_df.loc[j,(ogwin,"EV")]=0
    
    # change overflow
    overflow=overflow-state_voters_moved
    total_voters_moved=total_voters_moved+state_voters_moved # update total voters moved
    
    print(f"Flipped {target_state.loc['States'].to_string()[4:]} by moving {state_voters_moved} voters from {source_state.loc['States'].to_string()[4:]}. Total voters moved: {total_voters_moved}. Flipped EV: {flipped_EV}. Overflow: {overflow}.")

    
    # move on to the next target state
    j=j+1
    target_state=ol_df.iloc[j] 

  else: # store overflow in pover and move on to the next source state
    print(f"Not enough voters in {source_state.loc['States'].to_string()[4:]} to flip {target_state.loc['States'].to_string()[4:]}. Overflow: {overflow}. Moving on to the next source state.")
    pover=overflow
    i+=1 
    source_state=ow_df.iloc[i] if i<sources else None
    overflow=source_state.loc[("Margin","Votes")]*(-1)-mini_margin+pover if i<sources else None # update overflow for the new source state




# ol_df

Flipped Wisconsin by moving 30397 voters from California. Total voters moved: 30397. Flipped EV: 10. Overflow: 3163085.
Flipped Michigan by moving 81103 voters from California. Total voters moved: 111500. Flipped EV: 25. Overflow: 3081982.
Flipped Pennsylvania by moving 121266 voters from California. Total voters moved: 232766. Flipped EV: 44. Overflow: 2960716.
Flipped Georgia by moving 116100 voters from California. Total voters moved: 348866. Flipped EV: 60. Overflow: 2844616.
Flipped Nevada by moving 47008 voters from California. Total voters moved: 395874. Flipped EV: 66. Overflow: 2797608.
Flipped North Carolina by moving 184048 voters from California. Total voters moved: 579922. Flipped EV: 82. Overflow: 2613560.
Flipped Alaska by moving 45432 voters from California. Total voters moved: 625354. Flipped EV: 85. Overflow: 2568128.
Flipped Arizona by moving 188382 voters from California. Total voters moved: 813736. Flipped EV: 96. Overflow: 2379746.
Flipped Montana by moving 121173

In [37]:
# update the source states
ow_df=losers_df.copy(deep=True)

i=0
reimburse = total_voters_moved
while i<sources:
    if reimburse>ow_df.loc[i,("Margin","Votes")]*(-1)-mini_margin:
        state_voters_removed=ow_df.loc[i,("Margin","Votes")]*(-1)-mini_margin

        # change votes
        ow_df.loc[i,(oglos,"Votes")]=int(ow_df.loc[i,(oglos,"Votes")])-state_voters_removed
        ow_df.loc[i,"Total Votes"]=int(ow_df["Total Votes"].iloc[i])-state_voters_removed
        ow_df.loc[i,("Margin","Votes")]=mini_margin*(-1)

        reimburse-=state_voters_removed
        print(f"Removed {state_voters_removed} voters from {ow_df.loc[i,'States'].to_string()[4:]}. Remaining voters to reimburse: {reimburse}.")
    else:
        state_voters_removed=reimburse

        # change votes
        ow_df.loc[i,(oglos,"Votes")]=int(ow_df.loc[i,(oglos,"Votes")])-state_voters_removed
        ow_df.loc[i,"Total Votes"]=int(ow_df["Total Votes"].iloc[i])-state_voters_removed
        ow_df.loc[i,("Margin","Votes")]=ow_df.loc[i,("Margin","Votes")]+state_voters_removed

        reimburse-=state_voters_removed
        print(f"Removed {state_voters_removed} voters from {ow_df.loc[i,'States'].to_string()[4:]}. All voters reimbursed.")
    i+=1
        
# ow_df

Removed 2054990 voters from California. All voters reimbursed.


In [38]:
# concatenate the winner and loser
final_df=pd.concat([ol_df,ow_df]).sort_values(by=[("Margin","Votes")])

#update percentages
final_df.loc[:,(oglos,"%")]=(final_df.loc[:,(oglos,"Votes")]/final_df.loc[:,"Total Votes"]).astype(float).round(4)*100
final_df.loc[:,(ogwin,"%")]=(final_df.loc[:,(ogwin,"Votes")]/final_df.loc[:,"Total Votes"]).astype(float).round(4)*100
final_df.loc[:,("Margin","%")]=(final_df.loc[:,("Margin","Votes")]/final_df.loc[:,"Total Votes"]).astype(float).round(4)*100

final_df

States Trump/Vance Republican            Harris/Walz Democratic  \
                                    Votes      %  EV                  Votes   
0       California                6081697  44.04   0                7221189   
1         New York                3578899  43.31   0                4619195   
2    Massachusetts                1251303  36.02   0                2126518   
3         Maryland                1035550  34.08   0                1902577   
4       Washington                1530923  39.01   0                2245849   
5         Illinois                2449079  43.47   0                3062863   
6         Colorado                1377441  43.14   0                1728159   
7           Oregon                 919480  40.97   0                1240600   
8             D.C.                  21076   6.47   0                 294185   
9         Virginia                2075085  46.05   0                2335395   
10     Connecticut                 736918  41.89   0                 992053   
11      New Jersey                1968215  46.06   0                2220713   
12       Minnesota                1519032  46.68   0                1656979   
13          Hawaii                 193661  37.48   0                 313044   
14         Vermont                 119395  32.32   0                 235791   
15            ME-1                 165214  38.09   0                 258863   
16        Delaware                 214351  41.79   0                 289758   
17    Rhode Island                 214406  41.76   0                 285156   
18           Maine                 377977  45.46   0                 435652   
19      New Mexico                 423391  45.85   0                 478802   
20   New Hampshire                 395523  47.87   0                 418488   
21            NE-2                 148905  46.73   0                 163541   
0        Wisconsin                1697626  49.16   0                1698626   
4           Nevada                 751205  49.04   0                 752205   
8          Montana                 352079  48.62   0                 353079   
9           Kansas                 758802  49.19   0                 759802   
10            ME-2                 212763  48.95   0                 213763   
11            Iowa                 927019   49.2   0                 928019   
12            Ohio                3180116  49.57   0                3181116   
5   North Carolina                2898423  49.27   0                2899423   
7          Arizona                1770242  49.47   0                1771242   
6           Alaska                 184458  48.08   0                 185458   
1         Michigan                2816636  49.03   0                2817636   
2     Pennsylvania                3543308  49.35   0                3544308   
3          Georgia                2663117  49.62   0                2664117   
15            NE-1                 177666  55.49   1                 136153   
14         Wyoming                 192633   71.6   3                  69527   
16    South Dakota                 272081  63.43   3                 146859   
17    North Dakota                 246505  66.96   3                 112327   
33            NE-3                 238245  76.03   1                  70301   
32        Nebraska                 564816  59.32   2                 369995   
18     Mississippi                 747744  60.89   6                 466668   
29   West Virginia                 533556  69.97   4                 214309   
22            Utah                 883818  59.38   6                 562566   
30           Idaho                 605246  66.87   4                 274972   
25        Arkansas                 759241   64.2   6                 396905   
24       Louisiana                1208505  60.22   8                 766870   
20  South Carolina                1483747  58.23   9                1028452   
26        Oklahoma                1036213  66.16   7          

# Final Choropleth Maps (in dev)

# Remaining objectives/to do


Overflow algo: completed
* finished minimum EV, multiple source states

Underflow algo: 
*   minimum EV and multiple sources

Choropleth graphs


# GENERALIZATION
Many features of the code need to be modified so that it can be run on any election, not just 2024
* Since different parties win different elections, references to column names need to be relaced with column indices.
  * The original column names can be put into a series that can be referenced later to specify the part/candidate names
  * After "states" the winner is typically the first column, then the loser, then the third parties, then margin, then margin swing, then total
